# OCR Hán — Kiểm tra chéo hai động cơ (PaddleOCR + CLC OCR) & Hiệu đính thủ công
## 01. Introduction
Notebook này triển khai quy trình **Dual-engine Han OCR Cross-check with Human-in-the-loop Correction**:
- **Primary OCR**: PaddleOCR (`PP-OCRv6_medium_det` + `PP-OCRv6_medium_rec` — giữ nguyên cấu hình cột dọc `vertical_rtl`, hàng ngang Lý–Trần `horizontal_ltr`, ngoại lệ hình học và tiền xử lý Bắc sứ `gray_scale`/`binary`).
- **Secondary OCR**: Kim Hán Nôm (CLC) API (`image-upload` + `image-ocr`, KHÔNG dùng endpoint `sentence-alignment`) hoặc chế độ import thủ công kết quả JSON/TXT có ánh xạ nguồn tường minh.
- **Cross-check**: Chuẩn hóa bản sao (`strict` & `content`), khớp vùng phân cấp, kiểm tra reading order, dóng dòng DP (`1-1`, `1-n`, `n-1`), dóng ký tự Levenshtein và phân loại bất đồng.
- **Manual Fixing**: Hiệu đính từng mục qua CSV/HTML/bảng xem trước; chỉ chèn `[MASK]` khi thành viên xác nhận `INSERT_MASK`; **không** chọn kết quả thắng theo confidence và **không** tự chuyển sang Text Cleaning.

Notebook cho OCR tiếng Việt được tách riêng tại `OCR_viet.ipynb`.

## 02. Environment Setup
Kiểm tra môi trường chạy (Windows/Linux/Kaggle/Colab), thiết bị CPU/GPU, cấu hình thư viện và nạp package `hanviet` mà không tải weight hay gọi API lúc import.

In [ ]:
import json
from pathlib import Path
import sys

ROOT_CANDIDATE = Path.cwd()
for cand in [ROOT_CANDIDATE, ROOT_CANDIDATE.parent, ROOT_CANDIDATE.parent.parent]:
    if (cand / "src" / "hanviet").is_dir():
        if str(cand / "src") not in sys.path:
            sys.path.insert(0, str(cand / "src"))
        break

from hanviet.common.paths import resolve_paths, detect_runtime, resolve_device
from hanviet.common.config import check_dependencies, load_method_config, load_corpus_config
from hanviet.common.io import read_json, write_json, sha256

PATHS = resolve_paths()
RUNTIME = detect_runtime("auto")
print("Project root:", PATHS.root)
print("Data root:", PATHS.data_root)
print("Output root:", PATHS.output_root)
print("Runtime:", RUNTIME)
print("Han OCR runtime deps:", check_dependencies("ocr_han"))

## 03. Configuration
Chọn `CORPUS_ID` (`CPO_09`, `CPO_10`, `CPO_11`, `CPO_12`), `MEMBER_ID`, danh sách tác phẩm/trang, và `MODE` chạy độc lập:
`preprocess` | `paddle_ocr` | `clc_ocr` | `import_clc` | `cross_check` | `review` | `apply_corrections` | `evaluate` | `export`.

In [ ]:
CORPUS_ID = "CPO_09"          # CPO_09 (BacSu), CPO_10 (Hich), CPO_11 (ThiDinh), CPO_12 (LyTran)
MEMBER_ID = "TV02"            # TV02 hoặc TV04
TOPIC_CODE = "CV06"

SELECTED_WORKS = []           # Ví dụ ["bai0024"] hoặc [] lấy tất cả trong phạm vi
SELECTED_PAGES = []           # Ví dụ [49, 50] hoặc [] lấy toàn bộ trang

PRIMARY_ENGINE = "paddleocr"
SECONDARY_ENGINE = "clc_ocr"

INPUT_ROOT = None             # None -> tự phân giải <data_root>/<BookFolder>/01_nguon/han
OUTPUT_ROOT = None            # None -> tự phân giải <output_root>/<MEMBER_ID>/<TOPIC>_<CORPUS_ID>/02_ocr_tho/han

DEVICE = resolve_device("auto")
MAX_IMAGES = 3                # Giới hạn mặc định an toàn khi kiểm tra nhanh; đặt None để chạy toàn bộ
RUN_ID = "han_dual_v01"

MODE = "cross_check"          # preprocess | paddle_ocr | clc_ocr | import_clc | cross_check | review | apply_corrections | evaluate | export

OVERWRITE = False
CALL_CLC_API = False          # Phải bật tường minh True mới gửi request thật tới CLC OCR API
EXPORT_APPROVED = False       # Chỉ bật True sau khi hoàn thành Manual Fixing toàn bộ vùng đã chọn

BAC_PREPROCESS_VARIANT = "soft_gray"   # original | gray_only | soft_gray | binary_otsu
LOW_CONFIDENCE_THRESHOLD = 0.60
USE_DP_LINE_MATCH = True

BOOK_FOLDER_MAP = {"CPO_09": "BacSu", "CPO_10": "HichTuongSi", "CPO_11": "ThiDinh", "CPO_12": "LyTran"}
BOOK_FOLDER = BOOK_FOLDER_MAP[CORPUS_ID]
OCR_CFG = load_method_config("ocr", PATHS.root)
CORPUS_CFG = load_corpus_config(CORPUS_ID, PATHS.root)

resolved_input = Path(INPUT_ROOT) if INPUT_ROOT else (PATHS.data_root / BOOK_FOLDER / "01_nguon" / "han")
resolved_output = Path(OUTPUT_ROOT) if OUTPUT_ROOT else (PATHS.output_root / MEMBER_ID / f"{TOPIC_CODE}_{CORPUS_ID}" / "02_ocr_tho" / "han")

from hanviet.extraction.exporter import stage_han_dirs
STAGE_DIRS = stage_han_dirs(resolved_output)
print({"mode": MODE, "corpus": CORPUS_ID, "device": DEVICE, "input": str(resolved_input), "output": str(STAGE_DIRS["root"])})

## 04. Source Inventory
Kiểm kê ảnh PNG / TXT nguồn trong phạm vi được chọn (không chạy OCR cho các nguồn PDF Text hoặc TXT thuần của Hịch / Bắc sứ Việt).

In [ ]:
import re

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff", ".bmp"}

def discover_han_images(input_root: Path, selected_works: list[str], selected_pages: list[int], max_images: int | None):
    if not input_root.is_dir():
        print("Thư mục ảnh nguồn chưa tồn tại trên máy này:", input_root)
        return []
    items = []
    for p in sorted(input_root.rglob("*")):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS:
            continue
        work_id = p.parent.name
        if selected_works and not any(w in work_id or w == work_id for w in selected_works):
            continue
        m = re.search(r"_pdfp(\d+)\.", p.name)
        pdf_page = int(m.group(1)) if m else None
        if selected_pages and pdf_page not in selected_pages:
            continue
        items.append({
            "work_id": work_id,
            "source_id": f"{work_id}/{p.stem}",
            "part_stem": p.stem,
            "pdf_page": pdf_page,
            "image_path": p,
            "input_sha256": sha256(p),
        })
    return items[:max_images] if max_images else items

SOURCE_ITEMS = discover_han_images(resolved_input, SELECTED_WORKS, SELECTED_PAGES, MAX_IMAGES)
print(f"Tìm thấy {len(SOURCE_ITEMS)} ảnh Hán hợp lệ trong phạm vi.")
for item in SOURCE_ITEMS[:5]:
    print(" -", item["source_id"], "| pdf_page=", item["pdf_page"], "| sha=", item["input_sha256"][:12])

## 05. Image Preprocessing
Chỉ chạy khi `MODE == "preprocess"` (áp dụng khử dấu triện đỏ, cân nền giấy và `soft_gray` / `binary_otsu` cho Bắc sứ `CPO_09` theo đúng `bac_su_preprocess`, giữ nguyên tọa độ 1:1).

In [ ]:
from hanviet.extraction.preprocessing.han_preprocess import bac_su_preprocess, BAC_MAIN_PREPROCESS_PARAMS_DEFAULT

if MODE == "preprocess" and SOURCE_ITEMS:
    from PIL import Image
    pre_dir = STAGE_DIRS["paddle_vis"] / "preprocessed"
    pre_dir.mkdir(parents=True, exist_ok=True)
    for item in SOURCE_ITEMS:
        with Image.open(item["image_path"]) as img:
            variants, stats = bac_su_preprocess(img, BAC_MAIN_PREPROCESS_PARAMS_DEFAULT)
        out_png = pre_dir / f"{item['part_stem']}_{BAC_PREPROCESS_VARIANT}.png"
        variants[BAC_PREPROCESS_VARIANT].save(out_png, format="PNG")
        print("Saved preprocessed:", out_png.name, "red_mask_frac=", round(stats["red_mask_fraction"], 4))
else:
    print(f"Bỏ qua bước preprocess (MODE={MODE!r}).")

## 06. PaddleOCR (Primary Engine)
Chỉ chạy khi `MODE == "paddle_ocr"`. Kết quả thô được lưu độc lập tại `02_ocr_tho/han/paddle_ocr/` kèm `manifest.json` và overlay vùng.

In [ ]:
from hanviet.extraction.engines.paddle_han import PaddleHanEngine
from hanviet.extraction.exporter import save_independent_engine_outputs

if MODE == "paddle_ocr" and SOURCE_ITEMS:
    paddle_cfg = OCR_CFG.get("han", {}).get("paddle", {})
    layout_mode = "horizontal_ltr" if CORPUS_ID == "CPO_12" else "vertical_rtl"
    engine = PaddleHanEngine(
        detection_model=paddle_cfg.get("detection_model", "PP-OCRv6_medium_det"),
        recognition_model=paddle_cfg.get("recognition_model", "PP-OCRv6_medium_rec"),
        inference_engine=paddle_cfg.get("inference_engine", "transformers"),
        device=DEVICE,
        batch_size=int(paddle_cfg.get("batch_size", 1)),
        rec_score_threshold=float(paddle_cfg.get("rec_score_threshold", 0.0)),
    )
    for item in SOURCE_ITEMS:
        meta_json = STAGE_DIRS["paddle_meta"] / f"{item['part_stem']}.records.json"
        if meta_json.is_file() and not OVERWRITE:
            print("Skip existing PaddleOCR result:", item["part_stem"])
            continue
        recs = engine.recognize_image(
            item["image_path"],
            source_id=item["source_id"],
            work_id=item["work_id"],
            corpus_id=CORPUS_ID,
            member_id=MEMBER_ID,
            pdf_page=item["pdf_page"],
            layout=layout_mode,
            preprocess_variant=BAC_PREPROCESS_VARIANT if CORPUS_ID == "CPO_09" else None,
            run_id=RUN_ID,
        )
        save_independent_engine_outputs(
            STAGE_DIRS["root"] / "paddle_ocr",
            engine_name="paddleocr",
            records=[r.to_dict() for r in recs],
            part_stem=item["part_stem"],
            image_path=item["image_path"],
            render_overlay=True,
        )
        print("Saved PaddleOCR:", item["part_stem"], "lines=", len(recs))
else:
    print(f"Bỏ qua bước PaddleOCR inference (MODE={MODE!r}).")

## 07. CLC OCR (Secondary Engine & Manual Import)
- `MODE == "clc_ocr"` (kèm `CALL_CLC_API = True`): gửi ảnh qua adapter `CLCOCRAdapter` (`image-upload` + `image-ocr`), tôn trọng rate limit & cache theo SHA-256.
- `MODE == "import_clc"`: nạp kết quả JSON/TXT đã xuất từ công cụ CLC kèm ánh xạ `source_id`, `work_id`, `pdf_page` tường minh (không đoán theo tên file).

In [ ]:
from hanviet.extraction.engines.clc_han import CLCOCRAdapter, import_clc_ocr_json, import_clc_ocr_text

CLC_IMPORT_MAPPING = [
    # Ví dụ cho chế độ import_clc (điền đường dẫn thật khi import thủ công):
    # {"path": Path("path/to/clc_result.json"), "source_id": "bai0024/phan0001_pdfp0608", "work_id": "bai0024", "part_stem": "phan0001_pdfp0608", "pdf_page": 608, "kind": "json"}
]

if MODE == "clc_ocr" and SOURCE_ITEMS:
    if not CALL_CLC_API:
        print("CALL_CLC_API=False: giữ an toàn không gửi request CLC API; chỉ đọc cache nếu có.")
    clc_cfg = OCR_CFG.get("han", {}).get("clc_ocr", {})
    clc = CLCOCRAdapter(
        options=clc_cfg.get("default_options"),
        project_root=PATHS.root,
    )
    for item in SOURCE_ITEMS:
        cache_path = STAGE_DIRS["clc_resp"] / f"{item['part_stem']}.json"
        if not CALL_CLC_API and not cache_path.is_file():
            print("API_NOT_CALLED (no cache):", item["part_stem"])
            continue
        recs = clc.recognize_image(
            item["image_path"],
            source_id=item["source_id"],
            work_id=item["work_id"],
            corpus_id=CORPUS_ID,
            member_id=MEMBER_ID,
            pdf_page=item["pdf_page"],
            cache_meta_path=cache_path,
            error_log_path=STAGE_DIRS["clc_meta"] / "clc_errors.jsonl",
            run_id=RUN_ID,
        )
        save_independent_engine_outputs(
            STAGE_DIRS["root"] / "clc_ocr",
            engine_name="clc_ocr",
            records=[r.to_dict() for r in recs],
            part_stem=item["part_stem"],
        )
        print("Saved CLC OCR:", item["part_stem"], "status=", recs[0].status if recs else "EMPTY")
elif MODE == "import_clc":
    for spec in CLC_IMPORT_MAPPING:
        fpath = Path(spec["path"])
        if spec.get("kind") == "txt":
            recs = import_clc_ocr_text(
                fpath,
                source_id=spec["source_id"],
                work_id=spec["work_id"],
                corpus_id=CORPUS_ID,
                member_id=MEMBER_ID,
                pdf_page=int(spec["pdf_page"]),
                run_id=RUN_ID,
            )
        else:
            recs = import_clc_ocr_json(
                fpath,
                source_id=spec["source_id"],
                work_id=spec["work_id"],
                corpus_id=CORPUS_ID,
                member_id=MEMBER_ID,
                pdf_page=spec.get("pdf_page"),
                run_id=RUN_ID,
            )
        save_independent_engine_outputs(
            STAGE_DIRS["root"] / "clc_ocr",
            engine_name="clc_ocr",
            records=[r.to_dict() for r in recs],
            part_stem=spec["part_stem"],
        )
        print("Imported CLC:", spec["part_stem"], "records=", len(recs))
else:
    print(f"Bỏ qua bước CLC OCR (MODE={MODE!r}).")

## 08. Load Existing OCR Results
Nạp riêng các bản ghi `OCRRecord` đã lưu từ `paddle_ocr/metadata/` và `clc_ocr/metadata/` để sẵn sàng kiểm tra chéo ngoại tuyến.

In [ ]:
from hanviet.extraction.schema import OCRRecord

def load_engine_records(meta_dir: Path) -> list[OCRRecord]:
    records: list[OCRRecord] = []
    if not meta_dir.is_dir():
        return records
    for f in sorted(meta_dir.glob("*.records.json")):
        payload = read_json(f)
        for item in payload.get("records", []):
            records.append(OCRRecord(**{k: v for k, v in item.items() if k != "extra"}, extra=item.get("extra", {})))
    return records

PRIMARY_RECORDS = load_engine_records(STAGE_DIRS["paddle_meta"])
SECONDARY_RECORDS = load_engine_records(STAGE_DIRS["clc_meta"])
print(f"Loaded Primary (PaddleOCR): {len(PRIMARY_RECORDS)} | Secondary (CLC OCR): {len(SECONDARY_RECORDS)}")

## 09. Region Matching, 10. Reading Order Validation, 11. Line Matching & 12. Character-level Cross-check
Thực hiện toàn bộ luồng Cross-check phân cấp (`Corpus -> Work -> Source -> Page -> Region -> Line/DP -> Character Levenshtein`) và phân loại bất đồng có cấu trúc.

In [ ]:
from hanviet.extraction.cross_check import cross_check_records
from hanviet.extraction.exporter import export_cross_check_bundle

CROSS_CHECK_REPORT = None
cross_check_json_path = STAGE_DIRS["cross_check"] / "cross_check_report.json"

if MODE in {"cross_check", "review", "apply_corrections", "evaluate", "export"}:
    if MODE == "cross_check" and (PRIMARY_RECORDS or SECONDARY_RECORDS):
        CROSS_CHECK_REPORT = cross_check_records(
            PRIMARY_RECORDS,
            SECONDARY_RECORDS,
            variant="han",
            use_dp_line_match=USE_DP_LINE_MATCH,
            low_confidence_threshold=LOW_CONFIDENCE_THRESHOLD,
        )
        write_json(cross_check_json_path, CROSS_CHECK_REPORT)
        bundle = export_cross_check_bundle(STAGE_DIRS["cross_check"], CROSS_CHECK_REPORT)
        print("Cross-check summary:", CROSS_CHECK_REPORT["status"], CROSS_CHECK_REPORT["summary"])
        print("Reading order warnings:", len(CROSS_CHECK_REPORT.get("reading_order_warnings", [])))
        print("Disagreements logged:", len(CROSS_CHECK_REPORT.get("disagreements", [])))
    elif cross_check_json_path.is_file():
        CROSS_CHECK_REPORT = read_json(cross_check_json_path)
        print("Loaded saved cross_check_report.json:", CROSS_CHECK_REPORT.get("status"), CROSS_CHECK_REPORT.get("summary"))
    else:
        print("Chưa có dữ liệu PaddleOCR/CLC OCR để cross-check.")

## 13. Disagreement Visualization
Hiển thị bảng `pandas`, các phép toán ký tự `[SUB:...]`, `[DEL:...]`, `[INS:...]` và bản HTML đối chiếu trực quan ngay trong notebook.

In [ ]:
import pandas as pd
from IPython.display import HTML, display
from hanviet.extraction.manual_review import build_ocr_review_table, render_ocr_diff_html

if CROSS_CHECK_REPORT and CROSS_CHECK_REPORT.get("items"):
    REVIEW_ROWS = build_ocr_review_table(CROSS_CHECK_REPORT)
    df_review = pd.DataFrame(REVIEW_ROWS)
    display(df_review[["item_id", "work_id", "pdf_page", "match_type", "status", "primary_text", "secondary_text", "char_diff_summary", "issue_types"]].head(30))
    display(HTML(render_ocr_diff_html(CROSS_CHECK_REPORT)))
else:
    REVIEW_ROWS = []
    print("Không có mục cross-check để hiển thị.")

## 14. Manual Fixing & 15. Apply Approved Corrections
- Sửa file `manual_review/review_queue.csv` bằng tay (hoặc Excel), điền `decision` (`ACCEPT_PRIMARY`, `ACCEPT_SECONDARY`, `MANUAL_EDIT`, `KEEP_BOTH_AGREED`, `INSERT_MASK`, `FIX_READING_ORDER`, `REPROCESS`, `REQUEST_SOURCE_REVIEW`, `DEFER`), `corrected_text`, `reason`, `reviewed_by`.
- `[MASK]` **chỉ** được chèn khi có quyết định `INSERT_MASK` / `MANUAL_EDIT` từ thành viên.
- Khi `MODE == "apply_corrections"`, đọc CSV đã duyệt và áp dụng an toàn (không ghi đè OCR thô hay bản đã duyệt).

In [ ]:
import csv
from hanviet.extraction.manual_review import apply_manual_ocr_decisions, export_ocr_review_csv

review_queue_csv = STAGE_DIRS["manual_review"] / "review_queue.csv"
decisions_csv = STAGE_DIRS["manual_review"] / "review_decisions.csv"
reviewed_json_path = STAGE_DIRS["manual_review"] / f"{RUN_ID}_reviewed.json"
MANUAL_RESULT = None

if MODE == "review" and REVIEW_ROWS and not review_queue_csv.is_file():
    export_ocr_review_csv(review_queue_csv, REVIEW_ROWS)
    print("Đã tạo hàng đợi review tại:", review_queue_csv)
elif MODE == "apply_corrections" and CROSS_CHECK_REPORT:
    source_csv = decisions_csv if decisions_csv.is_file() else review_queue_csv
    if source_csv.is_file():
        with source_csv.open(encoding="utf-8-sig", newline="") as f:
            decisions = list(csv.DictReader(f))
        MANUAL_RESULT = apply_manual_ocr_decisions(
            CROSS_CHECK_REPORT,
            decisions,
            corrected_text_path=STAGE_DIRS["corrected_merged"] / f"{CORPUS_ID}_{MEMBER_ID}_{RUN_ID}.txt",
            correction_log_path=STAGE_DIRS["manual_review"] / "correction_log.jsonl",
            reviewed_json_path=reviewed_json_path,
            raw_protected_roots=[STAGE_DIRS["root"] / "paddle_ocr", STAGE_DIRS["root"] / "clc_ocr"],
            reviewer=MEMBER_ID,
            allow_overwrite_approved=OVERWRITE,
        )
        print("Manual review result:", MANUAL_RESULT["review_status"], "reviewed=", MANUAL_RESULT["reviewed_count"], "deferred=", MANUAL_RESULT["deferred_count"])
    else:
        print("Chưa tìm thấy CSV quyết định tại:", source_csv)
elif reviewed_json_path.is_file():
    MANUAL_RESULT = read_json(reviewed_json_path)

## 16. Assemble Reviewed Transcription & 18. Export Results (Text Cleaning Compatible)
Xuất bản chép đã duyệt thành định dạng tương thích hoàn toàn với Stage 03 (`clean_text.ipynb` / `hanviet.cleaning.pipeline.read_sources`). Không chạy Text Cleaning tự động tại đây.

In [ ]:
from hanviet.extraction.exporter import export_cleaning_compatible_han_transcription

if MODE == "export" and MANUAL_RESULT:
    if not MANUAL_RESULT.get("approved") and not EXPORT_APPROVED:
        print("Cảnh báo: Chưa xác nhận duyệt đủ (approved=False). Đặt EXPORT_APPROVED=True nếu muốn xuất nháp.")
    else:
        # Gom các dòng đã duyệt theo từng work_id / part
        by_part = {}
        for it, rec in zip(CROSS_CHECK_REPORT.get("items", []), MANUAL_RESULT.get("records", [])):
            if rec.get("final_text") is None:
                continue
            src = (it.get("primary") or it.get("secondary") or {})
            work = src.get("work_id", "unknown_work")
            img_name = Path(src.get("image_reference", "part0001.png")).name
            by_part.setdefault((work, img_name), {"lines": [], "sha": src.get("input_sha256", "")})["lines"].append(rec["final_text"])
        for work in {w for w, _ in by_part}:
            parts = [
                {"image_name": img_name, "text": "\n".join(info["lines"]), "source_sha256": info["sha"]}
                for (w, img_name), info in sorted(by_part.items()) if w == work
            ]
            out = export_cleaning_compatible_han_transcription(
                STAGE_DIRS["corrected_parts"].parent / "cleaning_projection",
                corpus_id=CORPUS_ID,
                work_stem=work,
                part_transcriptions=parts,
                approved=bool(MANUAL_RESULT.get("approved")),
                allow_draft=not bool(MANUAL_RESULT.get("approved")),
            )
            print("Exported Stage 03-compatible Han transcription for", work, "->", out.get("merged"))
else:
    print(f"Bỏ qua bước export Text Cleaning projection (MODE={MODE!r}).")

## 17. OCR Evaluation
Tính toán độ đồng thuận (`strict` & `content`), độ phủ, thống kê manual fixing, khung thí nghiệm `H01`–`H06`, và CER/WER vi mô + vĩ mô khi có bản chuẩn kiểm chứng (`gold_references`).

In [ ]:
from hanviet.evaluation.metrics import compute_han_ocr_evaluation, experiment_matrix_template

GOLD_REFERENCES = None   # Đặt list[str] khớp số lượng CROSS_CHECK_REPORT['items'] nếu có gold dataset

if MODE in {"evaluate", "cross_check", "apply_corrections"} and CROSS_CHECK_REPORT:
    EVAL_METRICS = compute_han_ocr_evaluation(
        CROSS_CHECK_REPORT,
        MANUAL_RESULT,
        gold_references=GOLD_REFERENCES,
    )
    write_json(STAGE_DIRS["root"] / "evaluation" / "metrics.json", EVAL_METRICS)
    print(json.dumps(EVAL_METRICS, ensure_ascii=False, indent=2))
    display(pd.DataFrame(experiment_matrix_template()))

## 19. Limitations
- Hai engine PaddleOCR và CLC OCR đồng thuận (`AGREED_UNVERIFIED`) **không đồng nghĩa** văn bản chắc chắn đúng (false agreement vẫn có thể xảy ra trên chữ mờ/chữ hiếm).
- Confidence giữa PaddleOCR và CLC OCR không thể so sánh trực tiếp để chọn kết quả thắng (CLC `image-ocr` hiện không trả confidence).
- Tài liệu cổ (đặc biệt `CPO_09` Bắc sứ) chứa nhiều chữ Hán phồn thể/dị thể tự/chữ hiếm; Cross-check chỉ làm công cụ trợ giúp định vị điểm nghi ngờ, **Manual Fixing bởi chuyên viên là bắt buộc** trước khi chuyển sang Stage 03 Text Cleaning.